#Comparison test results with the GPT-3.5-Turbo-0125 model

##Intent Detection Results

In [1]:
!pip install openai==0.28
import openai
import os
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score
import re

openai.api_key = 'sk-GTLuuepLSvN411RRG90_saWY9is9e5fzYVfiOR-BRqT3BlbkFJNoPZs7x_RBiIEa_VSjC-bie1daScizpMfKm7hj3CEA'

def classify_intent(user_input):

    response = openai.ChatCompletion.create(
        model="ft:gpt-3.5-turbo-0125:personal:last:AD22qZqV",
        messages=[
          {"role": "system", "content": "You are a helpful assistant."},
          {"role": "user", "content": user_input}
        ]
)

    return response['choices'][0]['message']['content'].strip()

test_data = [
 {"question": "Find cases by R Sharma", "intent": "retrieve_case01", "entities": {"Author": ["R Sharma"]}},
    {"question": "Show cases with M Patel", "intent": "retrieve_case01", "entities": {"Author": ["M Patel"]}},
    {"question": "Retrieve cases on 23 july, 2002 with IPC Section 132", "intent": "retrieve_case01", "entities": { "wit$datetime": ["23 july, 2002"], "Statute": ["IPC Section 132"] }},
    {"question": "Search cases with A Verma, M Prasad where sentence is upheld", "intent": "retrieve_case01", "entities": {"Bench": ["A Verma, M Prasad"], "Verdict": ["sentence is upheld"]}},
    {"question": "Cases by L Yadav", "intent": "retrieve_case01", "entities": {"Author": ["L Yadav"]}},

    {"question": "Find statutes where Appeal dismissed", "intent": "retrieve_statute03", "entities": {"Verdict": ["Appeal dismissed"]}},
    {"question": "Show statutes on 19 June, 2001", "intent": "retrieve_statute03", "entities": {"wit$datetime": ["19 June, 2001"]}},
    {"question": "Retrieve statute of case by M Patel", "intent": "retrieve_statute03", "entities": {"Author": ["M Patel"]}},
    {"question": "Search statute by S Patel on 23 march, 2001", "intent": "retrieve_statute03", "entities": {"Author": ["S Patel"], "wit$datetime": ["23 march, 2001"]}},
    {"question": "Fetch statutes by S Patel", "intent": "retrieve_statute03", "entities": {"Author": ["S Patel"]}},

    {"question": "Find date where Appeal dismissed", "intent": "retrieve_date04", "entities": {"Verdict": ["Appeal dismissed"]}},
    {"question": "Show date with IPC Section 119", "intent": "retrieve_date04", "entities": {"Statute": ["IPC Section 119"]}},
    {"question": "Retrieve date where The order passed by the High Court is set aside with Income Tax Act", "intent": "retrieve_date04", "entities": {"Verdict": ["The order passed by the High Court is set aside"],"Statute": ["Income Tax Act"]}},
    {"question": "Search date with Patents Act by M Brown", "intent": "retrieve_date04", "entities": {"Author": ["M Brown"], "Statute": ["Patents Act"]}},
    {"question": "Fetch date by J Martin", "intent": "retrieve_date04", "entities": {"Author": ["J Martin"]}},
    {"question": "Show date with B Smith, L Wilson", "intent": "retrieve_date04", "entities": {"Bench": ["B Smith, L Wilson"]}},
    {"question": "Retrieve date where The appeal is allowed and the order passed by the High Court is set aside", "intent": "retrieve_date04", "entities": {"Verdict": ["The appeal is allowed and The order passed by the High Court is set aside"]}},
    {"question": "Find date The People vs. John Smith", "intent": "retrieve_date04", "entities": {"Case_Name": ["The People vs. John Smith"]}},

    {"question": "Find author where Appeal dismissed with IPC Section 30", "intent": "retrieve_author05", "entities": {"Statute": ["IPC Section 30"], "Verdict": ["Appeal dismissed"]}},
    {"question": "Show author of case with IPC Section 119", "intent": "retrieve_author05", "entities": {"Statute": ["IPC Section 119"]}},
    {"question": "Retrieve author of case where The order passed by the High Court is set aside with Income Tax Act", "intent": "retrieve_author05", "entities": {"Verdict": ["The order passed by the High Court is set aside"],"Statute": ["Income Tax Act"]}},
    {"question": "Search author of case based on Patents Act by M Brown", "intent": "retrieve_author05", "entities": {"Author": ["M Brown"], "Statute": ["Patents Act"]}},
    {"question": "Show author of case with B Smith, L Wilson", "intent": "retrieve_author05", "entities": {"Bench": ["B Smith", "L Wilson"]}},
    {"question": "Retrieve author of case on 25 September, 2026", "intent": "retrieve_author05", "entities": {"wit$datetime": ["25 September, 2026"]}},
    {"question": "Find author of case The People vs. John Smith", "intent": "retrieve_author05", "entities": {"Case_Name": ["The People vs. John Smith"]}},

    {"question": "Find bench where Appeal dismissed", "intent": "retrieve_bench06", "entities": {"Verdict": ["Appeal dismissed"]}},
    {"question": "Show bench with IPC Section 119", "intent": "retrieve_bench06", "entities": {"Statute": ["IPC Section 119"]}},
    {"question": "Retrieve bench where The order passed by the High Court is set aside with Income Tax Act", "intent": "retrieve_bench06", "entities": {"Statute": ["Income Tax Act"], "Verdict": ["The order passed by the High Court is set aside"]}},
    {"question": "Search bench based on Patents Act by M Brown", "intent": "retrieve_bench06", "entities": {"Author": ["M Brown"], "Statute": ["Patents Act"]}},
    {"question": "Fetch bench by J Martin", "intent": "retrieve_bench06", "entities": {"Author": ["J Martin"]}},
    {"question": "Retrieve bench on 25 September, 2026", "intent": "retrieve_bench06", "entities": {"wit$datetime": ["25 September, 2026"]}},
    {"question": "Find bench The People vs. John Smith", "intent": "retrieve_bench06", "entities": {"Case_Name": ["The People vs. John Smith"]}},

    {"question": "Find verdict by N Modi", "intent": "retrieve_verdict08", "entities": {"Author": ["N Modi"]}},
    {"question": "Show verdict with IPC Section 119", "intent": "retrieve_verdict08", "entities": {"Statute": ["IPC Section 119"]}},
    {"question": "Retrieve verdict following where Appeal dismissed with Income Tax Act", "intent": "retrieve_verdict08", "entities": {"Statute": ["Income Tax Act"],"Verdict": ["Appeal dismissed"]}},
    {"question": "Search verdict based on Patents Act by M Brown", "intent": "retrieve_verdict08", "entities": {"Author": ["M Brown"], "Statute": ["Patents Act"]}},
    {"question": "Fetch verdict by J Martin", "intent": "retrieve_verdict08", "entities": {"Author": ["J Martin"]}},
    {"question": "Show verdict with B Smith, L Wilson", "intent": "retrieve_verdict08", "entities": {"Author": ["B Smith", "L Wilson"]}},
    {"question": "Retrieve verdict on 25 September, 2026", "intent": "retrieve_verdict08", "entities": {"wit$datetime": ["25 September, 2026"]}},
    {"question": "Find verdict The People vs. John Smith", "intent": "retrieve_verdict08", "entities": {"Case_Name": ["The People vs. John Smith"]}},
    {"question": "Find cases with S Sharma and K Patel", "intent": "retrieve_verdict08", "entities": {"Author": ["S Sharma", "K Patel"]}},
    {"question": "Show cases with K Patel with IPC Section 13", "intent": "retrieve_verdict08", "entities": {"Author": ["K Patel"], "Statute": ["IPC Section 13"]}},
    {"question": "Retrieve cases with IPC Section 13 on 15th August, 2003", "intent": "retrieve_verdict08", "entities": {"Statute": ["IPC Section 13"], "wit$datetime": ["15th August, 2003"]}},
    {"question": "Search cases with P Verma, Q Prasad and N Patel on 20th March, 2002", "intent": "retrieve_verdict08", "entities": {"Bench": ["P Verma, Q Prasad and N Patel"], "wit$datetime": ["20th March, 2002"]}},
    {"question": "Cases by M Yadav with IPC Section 30", "intent": "retrieve_verdict08", "entities": {"Author": ["M Yadav"], "Statute": ["IPC Section 30"]}},

    {"question": "Find statutes where Sentence upheld", "intent": "retrieve_statute03", "entities": {"Verdict": ["Sentence upheld"]}},
    {"question": "Show statutes by S Patel on 15 July, 2001", "intent": "retrieve_statute03", "entities": {"Author": ["S Patel"], "wit$datetime": ["on 15 July, 2001"]}},
    {"question": "Retrieve statute of case by S Patel on 20th March, 2002", "intent": "retrieve_statute03", "entities": {"Author": ["S Patel"], "wit$datetime": ["20th March, 2002"]}},
    {"question": "Search statute by N Patel on 20th March, 2002", "intent": "retrieve_statute03", "entities": {"Author": ["N Patel"], "wit$datetime": ["20th March, 2002"]}},
    {"question": "Fetch statutes where sentence is upheld on 19 March, 2001", "intent": "retrieve_statute03", "entities": {"Verdict": ["sentence is upheld"], "wit$datetime": ["19 March, 2001"]}},

    {"question": "Find date where Sentence upheld with CrPC Section 136", "intent": "retrieve_date04", "entities": {"Statute": ["CrPC Section 136"],"Verdict": ["Sentence upheld"]}},
    {"question": "Show date with by P Brown IPC Section 136 and Evidence Act", "intent": "retrieve_date04", "entities": {"Author": ["P Brown"], "Statute": ["IPC Section 136", "Evidence Act"]}},
    {"question": "Retrieve date where The order passed by the High Court is set aside with Income Tax Act", "intent": "retrieve_date04", "entities": {"Statute": ["Income Tax Act"], "Verdict": ["The order passed by the High Court is set aside"]}},
    {"question": "Search date with Evidence Act by P Brown and Q Patel", "intent": "retrieve_date04", "entities": {"Bench": ["P Brown and Q Patel"], "Statute": ["Evidence Act"]}},
    {"question": "Fetch date by Q Patel, M Wilson, L Davis", "intent": "retrieve_date04", "entities": {"Bench": ["Q Patel", "M Wilson, L Davis"]}},
    {"question": "Show date with K Brown, L Wilson where The sentence is commuted", "intent": "retrieve_date04", "entities": {"Bench": ["K Brown, L Wilson"], "Verdict": ["The sentence is commuted"]}},
    {"question": "Retrieve date where The appeal is dismissed", "intent": "retrieve_date04", "entities": {"Verdict": ["The appeal is dismissed"]}},
    {"question": "Find date The State vs. John Doe with IPC Section 13", "intent": "retrieve_date04", "entities": {"Case_Name": ["The State vs. John Doe"], "Statute": ["IPC Section 13"]}},

    {"question": "Find author where Sentence upheld with CrPC Section 136", "intent": "retrieve_author05", "entities": {"Verdict": ["Sentence upheld"], "Statute": ["CrPC Section 136"]}},
    {"question": "Show author of case by P Brown with IPC Section 136 and Evidence Act", "intent": "retrieve_author05", "entities": {"Author": ["P Brown"], "Statute": ["IPC Section 136", "Evidence Act"]}},
    {"question": "Retrieve author of case where The order passed by the High Court is set aside with Income Tax Act", "intent": "retrieve_author05", "entities": {"Verdict": ["The order passed by the High Court is set aside"], "Statute": ["Income Tax Act"]}},
    {"question": "Search author of case based on Evidence Act", "intent": "retrieve_author05", "entities": {"Statute": ["Evidence Act"]}},
    {"question": "Show author of case with K Brown, L Wilson and IPC Section 136", "intent": "retrieve_author05", "entities": {"Bench": ["K Brown", "L Wilson"], "Statute": ["IPC Section 136"]}},
    {"question": "Retrieve author of case on 20th March, 2002 with IPC Section 13", "intent": "retrieve_author05", "entities": {"wit$datetime": ["20th March, 2002"], "Statute": ["IPC Section 13"]}},
    {"question": "Find author of case The State vs. John Doe with IPC Section 13", "intent": "retrieve_author05", "entities": {"Case_Name": ["The State vs. John Doe"], "Statute": ["IPC Section 13"]}},

    {"question": "Find bench with Sentence upheld and CrPC Section 136", "intent": "retrieve_bench06", "entities": {"Verdict": ["Sentence upheld"], "Statute": ["CrPC Section 136"]}},
    {"question": "Show bench with IPC Section 136 and Evidence Act", "intent": "retrieve_bench06", "entities": {"Statute": ["IPC Section 136", "Evidence Act"]}},
    {"question": "Retrieve bench where The order passed by the High Court is set aside with Income Tax Act", "intent": "retrieve_bench06", "entities": {"Verdict": ["The order passed by the High Court is set aside"], "Statute": ["IPC Section 13"]}},
    {"question": "Search bench based on Evidence Act by P Brown and Q Patel", "intent": "retrieve_bench06", "entities": {"Bench": ["P Brown", "Q Patel"], "Statute": ["Evidence Act"]}},
    {"question": "Fetch bench by Q Patel and M Wilson, L Davis", "intent": "retrieve_bench06", "entities": {"Bench": ["Q Patel", "M Wilson", "L Davis"]}},
    {"question": "Retrieve bench on 20th March, 2002 with IPC Section 13", "intent": "retrieve_bench06", "entities": {"wit$datetime": ["20th March, 2002"], "Statute": ["IPC Section 13"]}},
    {"question": "Find bench The State vs. John Doe with IPC Section 13", "intent": "retrieve_bench06", "entities": {"Case_Name": ["The State vs. John Doe"], "Statute": ["IPC Section 13"]}},

    {"question": "Find verdict with IPC Section 136", "intent": "retrieve_verdict08", "entities": {"Statute": ["IPC Section 136"]}},
    {"question": "Show verdict with IPC Section 136 by P Brown", "intent": "retrieve_verdict08", "entities": {"Statute": ["IPC Section 136"], "Author": ["P Brown"]}},
    {"question": "Retrieve verdict where Sentence upheld with Income Tax Act", "intent": "retrieve_verdict08", "entities": {"Statute": ["Income Tax Act"], "Verdict": ["Sentence upheld"]}},
    {"question": "Search verdict based on Evidence Act by P Brown and Q Patel", "intent": "retrieve_verdict08", "entities": {"Bench": ["P Brown and Q Patel"], "Statute": ["Evidence Act"]}},
    {"question": "Fetch verdict by Q Patel and M Wilson, L Davis", "intent": "retrieve_verdict08", "entities": {"Bench": ["Q Patel and M Wilson, L Davis"]}},
    {"question": "Show verdict with K Brown, L Wilson with IPC Section 136", "intent": "retrieve_verdict08", "entities": {"Bench": ["K Brown, L Wilson"], "Statute": ["IPC Section 136"]}},
    {"question": "Retrieve verdict on 20th March, 2002 with IPC Section 13", "intent": "retrieve_verdict08", "entities": {"Statute": ["IPC Section 13"], "wit$datetime": ["20th March, 2002"]}},
    {"question": "Find verdict The State vs. John Doe with IPC Section 13", "intent": "retrieve_verdict08", "entities": {"Case_Name": ["The State vs. John Doe"], "Statute": ["IPC Section 13"]}},
]

expected_intents = []
expected_entities = []
predictions = []
extracted_entities = []


for item in test_data:
    user_input = item["question"]
    expected_intent = item["intent"]
    expected_entity = item["entities"]

    output = classify_intent(user_input)

    # Extract intent from the output
    intent = output.split(" ")[1]  # Gets the second word which is the intent

    # Create formatted extracted entities
    entity_parts = output.split(" - ")

    for part in entity_parts[1:]:
        if ':' in part:  # Ensure it's a key-value pair
            key_value = part.split(":", 1)
            key = key_value[0].strip()
            value = key_value[1].strip() if len(key_value) > 1 else ""
            extracted_entities.append(f"{key}: {value}")

    predictions.append(intent)
    expected_intents.append(expected_intent)

    # Flatten expected entities in the same way as before
    for key, value in expected_entity.items():
        expected_entities.append(f"{key}: {' '.join(value)}")

intent_accuracy = accuracy_score(expected_intents, predictions)
intent_precision = precision_score(expected_intents, predictions, average='weighted', zero_division=0)
intent_recall = recall_score(expected_intents, predictions, average='weighted', zero_division=0)
intent_f1 = f1_score(expected_intents, predictions, average='weighted', zero_division=0)

expected_flat = expected_entities
extracted_flat = extracted_entities

# Ensure extracted_flat matches expected_flat in length
extracted_flat += [" "] * (len(expected_flat) - len(extracted_flat))

entity_accuracy = accuracy_score(expected_flat, extracted_flat)
entity_precision = precision_score(expected_flat, extracted_flat, average='weighted', zero_division=0)
entity_recall = recall_score(expected_flat, extracted_flat, average='weighted', zero_division=0)
entity_f1 = f1_score(expected_flat, extracted_flat, average='weighted', zero_division=0)

print(f"Intent Precision: {intent_precision:.2f}")
print(f"Intent Recall: {intent_recall:.2f}")
print(f"Intent F1 Score: {intent_f1:.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.57.4
    Uninstalling openai-1.57.4:
      Successfully uninstalled openai-1.57.4
Intent Precision: 0.97
Intent Recall: 0.91
Intent F1 Score: 0.93


##Entity Detection Results

In [2]:
from collections import Counter

# Sample extracted and expected lists
expected_entities = ['Author: R Sharma', 'Author: M Patel', 'wit$datetime: 23 july, 2002', 'Statute: IPC Section 132', 'Bench: A Verma, M Prasad', 'Verdict: sentence is upheld', 'Author: L Yadav', 'Verdict: Appeal dismissed', 'wit$datetime: 19 June, 2001', 'Author: M Patel', 'Author: S Patel', 'wit$datetime: 23 march, 2001', 'Author: S Patel', 'Verdict: Appeal dismissed', 'Statute: IPC Section 119', 'Verdict: The order passed by the High Court is set aside', 'Statute: Income Tax Act', 'Author: M Brown', 'Statute: Patents Act', 'Author: J Martin', 'Bench: B Smith, L Wilson', 'Verdict: The appeal is allowed and The order passed by the High Court is set aside', 'Case_Name: The People vs. John Smith', 'Statute: IPC Section 30', 'Verdict: Appeal dismissed', 'Statute: IPC Section 119', 'Verdict: The order passed by the High Court is set aside', 'Statute: Income Tax Act', 'Author: M Brown', 'Statute: Patents Act', 'Bench: B Smith L Wilson', 'wit$datetime: 25 September, 2026', 'Case_Name: The People vs. John Smith', 'Verdict: Appeal dismissed', 'Statute: IPC Section 119', 'Statute: Income Tax Act', 'Verdict: The order passed by the High Court is set aside', 'Author: M Brown', 'Statute: Patents Act', 'Author: J Martin', 'wit$datetime: 25 September, 2026', 'Case_Name: The People vs. John Smith', 'Author: N Modi', 'Statute: IPC Section 119', 'Statute: Income Tax Act', 'Verdict: Appeal dismissed', 'Author: M Brown', 'Statute: Patents Act', 'Author: J Martin', 'Author: B Smith L Wilson', 'wit$datetime: 25 September, 2026', 'Case_Name: The People vs. John Smith', 'Author: S Sharma K Patel', 'Author: K Patel', 'Statute: IPC Section 13', 'Statute: IPC Section 13', 'wit$datetime: 15th August, 2003', 'Bench: P Verma, Q Prasad and N Patel', 'wit$datetime: 20th March, 2002', 'Author: M Yadav', 'Statute: IPC Section 30', 'Verdict: Sentence upheld', 'Author: S Patel', 'wit$datetime: on 15 July, 2001', 'Author: S Patel', 'wit$datetime: 20th March, 2002', 'Author: N Patel', 'wit$datetime: 20th March, 2002', 'Verdict: sentence is upheld', 'wit$datetime: 19 March, 2001', 'Statute: CrPC Section 136', 'Verdict: Sentence upheld', 'Author: P Brown', 'Statute: IPC Section 136 Evidence Act', 'Statute: Income Tax Act', 'Verdict: The order passed by the High Court is set aside', 'Bench: P Brown and Q Patel', 'Statute: Evidence Act', 'Bench: Q Patel M Wilson, L Davis', 'Bench: K Brown, L Wilson', 'Verdict: The sentence is commuted', 'Verdict: The appeal is dismissed', 'Case_Name: The State vs. John Doe', 'Statute: IPC Section 13', 'Verdict: Sentence upheld', 'Statute: CrPC Section 136', 'Author: P Brown', 'Statute: IPC Section 136 Evidence Act', 'Verdict: The order passed by the High Court is set aside', 'Statute: Income Tax Act', 'Statute: Evidence Act', 'Bench: K Brown L Wilson', 'Statute: IPC Section 136', 'wit$datetime: 20th March, 2002', 'Statute: IPC Section 13', 'Case_Name: The State vs. John Doe', 'Statute: IPC Section 13', 'Verdict: Sentence upheld', 'Statute: CrPC Section 136', 'Statute: IPC Section 136 Evidence Act', 'Verdict: The order passed by the High Court is set aside', 'Statute: IPC Section 13', 'Bench: P Brown Q Patel', 'Statute: Evidence Act', 'Bench: Q Patel M Wilson L Davis', 'wit$datetime: 20th March, 2002', 'Statute: IPC Section 13', 'Case_Name: The State vs. John Doe', 'Statute: IPC Section 13', 'Statute: IPC Section 136', 'Statute: IPC Section 136', 'Author: P Brown', 'Statute: Income Tax Act', 'Verdict: Sentence upheld', 'Bench: P Brown and Q Patel', 'Statute: Evidence Act', 'Bench: Q Patel and M Wilson, L Davis', 'Bench: K Brown, L Wilson', 'Statute: IPC Section 136', 'Statute: IPC Section 13', 'wit$datetime: 20th March, 2002', 'Case_Name: The State vs. John Doe', 'Statute: IPC Section 13']

extracted_entities = ['Author: R Sharma', 'Author: M Patel', 'wit$datetime: 23 july, 2002 wit$statute: IPC Section 132', 'Bench: A Verma, M Prasad', 'Verdict: sentence is upheld', 'Author: L Yadav', 'Verdict: Appeal dismissed', 'wit$datetime: 19 June, 2001', 'Author: M Patel', 'Author: S Patel', 'Date: 23 march, 2001', 'Author: S Patel', 'Statute: IPC Section 119', 'Statute: Income Tax Act', 'Verdict: The order passed by the High Court is set aside', 'Statute: Patents Act', 'Author: M Brown', 'Author: J Martin', 'Bench: B Smith, L Wilson', 'Criteria: The appeal is allowed and the order passed by the High Court is set aside', 'Case_Name: The People vs. John Smith', 'Statute: IPC Section 30', 'Statute: IPC Section 119', 'Statute: Income Tax Act', 'Statute: Patents Act', 'Author: M Brown', 'Bench: B Smith, L Wilson', 'wit$datetime: 25 September, 2026', 'Statute: IPC Section 119', 'Statute: Income Tax Act', 'Verdict: The order passed by the High Court is set aside', 'Statute: Patents Act', 'Author: M Brown', 'Author: J Martin', 'wit$datetime: 25 September, 2026', 'Case_Name: The People vs. John Smith', 'Statute: IPC Section 119', 'Statute: Income Tax Act', 'Statute: Patents Act', 'Author: M Brown', 'Author: J Martin', 'Bench: B Smith, L Wilson', 'wit$datetime: 25 September, 2026', 'Case_Name: The People vs. John Smith', 'Bench: S Sharma, K Patel', 'Statute: IPC Section 13', 'Lawyer: K Patel', 'wit$datetime: 15th August, 2003', 'Statute: IPC Section 13', 'wit$datetime: 20th March, 2002', 'wit$contact: P Verma, Q Prasad, N Patel', 'Author: M Yadav', 'Statute: IPC Section 30', 'Statute: Sentence upheld', 'Author: S Patel', 'Date: 15 July, 2001', 'Author: S Patel', 'Date: 20th March, 2002', 'Author: N Patel', 'Date: 20th March, 2002', 'wit$datetime: 19 March, 2001', 'Statute: CrPC Section 136', 'Statute: IPC Section 136 and Evidence Act', 'Statute: Income Tax Act', 'Statute: Evidence Act', 'Authors: P Brown and Q Patel', 'Bench: Q Patel, M Wilson, L Davis', 'Bench: K Brown, L Wilson', 'Verdict: The sentence is commuted', 'Statute: IPC Section 13', 'Case: The State vs. John Doe', 'Statute: CrPC Section 136', 'Statute: IPC Section 136', 'Statute: Evidence Act', 'Statute: Income Tax Act', 'Statute: Evidence Act', 'Statute: IPC Section 136', 'Bench: K Brown, L Wilson', 'wit$datetime: 20th March, 2002', 'Statute: IPC Section 13', 'Precedent: Sentence upheld', 'Statute: CrPC Section 136', 'Statute: IPC Section 136 and Evidence Act', 'Statute: Income Tax Act', 'Verdict: The order passed by the High Court is set aside', 'Authors: P Brown and Q Patel', 'Statute: Evidence Act', 'Bench: Q Patel and M Wilson, L Davis', 'wit$datetime: 20th March, 2002', 'Statute: IPC Section 13', 'Statute: IPC Section 13', 'Case_Name: The State vs. John Doe', 'Statute: IPC Section 136', 'Statute: IPC Section 136', 'Judge: P Brown', 'Statute: Income Tax Act', 'Authors: P Brown and Q Patel', 'Statute: Evidence Act', 'Bench: Q Patel and M Wilson, L Davis', 'Bench: K Brown, L Wilson', 'Statute: IPC Section 136', 'wit$datetime: 20th March, 2002', 'Statute: IPC Section 13', 'Statute: IPC Section 13', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ']

# Count frequency in both lists
extracted_count = Counter(extracted_entities)
expected_count = Counter(expected_entities)

# Initialize a dictionary to hold frequencies
TP = 0
FP = 0
FN = 0

# Calculate True Positives and False Positives
for term in extracted_count:
    if term in expected_count:
        TP += min(extracted_count[term], expected_count[term])
    else:
        FP += extracted_count[term]

# Calculate False Negatives
for term in expected_count:
    if term not in extracted_count:
        FN += expected_count[term]

# Calculate precision, recall, and F1 score
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Print the metrics
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1_score:.2f}")


Precision: 0.67
Recall: 0.73
F1 Score: 0.70
